# GraphFleet Indexing Guide

This notebook demonstrates how to index your documents using GraphFleet, following GraphRAG's indexing approach. We'll cover:

1. Document preparation
2. Text chunking
3. Graph construction
4. Vector indexing

For more details, see [GraphRAG Indexing Documentation](https://microsoft.github.io/graphrag/index/overview/)

In [ ]:
import os
from pathlib import Path
from graphfleet.core import GraphFleet
from graphfleet.indexing import TextProcessor, GraphBuilder

# Set up project directory
project_dir = Path("./data")
raw_dir = project_dir / "raw"
processed_dir = project_dir / "processed"
index_dir = project_dir / "indexes"

# Create directories if they don't exist
for dir_path in [raw_dir, processed_dir, index_dir]:
    dir_path.mkdir(parents=True, exist_ok=True)

## 1. Document Preparation

First, let's prepare some example documents. GraphFleet supports various document formats including markdown, text, and PDF.

In [ ]:
# Create example documents
docs = [
    {
        "title": "introduction.md",
        "content": """
# Introduction to AI

Artificial Intelligence (AI) is a broad field of computer science focused on creating intelligent machines.
Key areas include:
- Machine Learning
- Natural Language Processing
- Computer Vision
"""
    },
    {
        "title": "machine_learning.md",
        "content": """
# Machine Learning

Machine Learning is a subset of AI that enables systems to learn from data.
Common types include:
1. Supervised Learning
2. Unsupervised Learning
3. Reinforcement Learning
"""
    }
]

# Write documents to raw directory
for doc in docs:
    with open(raw_dir / doc["title"], "w") as f:
        f.write(doc["content"])

## 2. Text Chunking

GraphFleet uses advanced text chunking strategies from GraphRAG to split documents into meaningful segments while preserving context.

In [ ]:
# Initialize text processor
processor = TextProcessor(
    chunk_size=512,  # Maximum chunk size in tokens
    overlap=50,      # Overlap between chunks
    split_method="markdown"  # Use markdown-aware splitting
)

# Process documents
chunks = processor.process_directory(raw_dir)

print(f"Generated {len(chunks)} chunks from {len(docs)} documents")
print("\nExample chunk:")
print(chunks[0])

## 3. Graph Construction

Now we'll build a knowledge graph from the chunks, following GraphRAG's graph construction approach.

In [ ]:
# Initialize graph builder
graph_builder = GraphBuilder(
    link_threshold=0.7,  # Similarity threshold for linking nodes
    max_neighbors=5      # Maximum neighbors per node
)

# Build graph
graph = graph_builder.build_graph(chunks)

print("Graph Statistics:")
print(f"Nodes: {graph.number_of_nodes()}")
print(f"Edges: {graph.number_of_edges()}")

## 4. Vector Indexing

Finally, we'll create vector indexes for efficient retrieval, using LanceDB as our vector store.

In [ ]:
# Initialize GraphFleet
graph_fleet = GraphFleet(project_dir)

# Create vector index
await graph_fleet.create_index(
    vector_store_type="lancedb",
    graph=graph,
    chunks=chunks
)

print("Vector index created successfully!")

## 5. Verify Indexing

Let's verify our index by performing a simple query.

In [ ]:
# Try a test query
result = await graph_fleet.query(
    "What are the main types of machine learning?",
    query_type="standard"
)

print("Query Result:")
print(result)

## Advanced Indexing Options

GraphFleet supports various advanced indexing options from GraphRAG:

In [ ]:
# Example: Custom indexing with advanced options
await graph_fleet.create_index(
    # Text processing options
    chunk_size=1024,
    overlap=100,
    split_method="markdown",
    
    # Graph construction options
    link_threshold=0.75,
    max_neighbors=8,
    edge_weight_method="cosine",
    
    # Vector store options
    vector_store_type="lancedb",
    dimension=1536,  # For text-embedding-3-small
    metric="cosine",
    
    # Additional options
    rebuild_index=True,  # Force rebuild existing index
    show_progress=True   # Show progress bar
)